In [0]:
from delta.tables import DeltaTable

In [0]:
# --------------------------------------------------
# 1. Tabela de destino
# --------------------------------------------------

destination = DeltaTable.forName(
    spark,
    "projeto2.silver.dados"
)

In [0]:
# --------------------------------------------------
# 2. Seleciona somente os registros novos
#    e pega a última versão de cada matrícula
# --------------------------------------------------

origin = spark.sql("""
    SELECT *
    FROM (
        SELECT
            b.*,
            ROW_NUMBER() OVER (
                PARTITION BY matricula
                ORDER BY ingestion_timestamp DESC
            ) AS rn
        FROM projeto2.bronze.dados b
        WHERE b.ingestion_timestamp > COALESCE((
                SELECT MAX(ingestion_timestamp) FROM projeto2.silver.dados), TIMESTAMP '1900-01-01'
            )
    )
    WHERE rn = 1
""")

In [0]:
display(origin)

In [0]:
# --------------------------------------------------
# 3. MERGE Bronze → Silver
# --------------------------------------------------

(
    destination.alias("destination")
    .merge(
        origin.alias("origin"),
        "destination.matricula = origin.matricula"
    )
    .whenMatchedUpdateAll(
        condition="origin.ingestion_timestamp > destination.ingestion_timestamp"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
spark.table("projeto2.silver.dados").display()